# Model construction

In this notebook, we construct the phenomenological (9-state), high-level (2-state), and mechanistic (38-state) gene expression models using BioCRNpyler and export their SBML files. 

In [1]:
import numpy as np
import biocrnpyler as bcp
from autoreduce import load_sbml
from sympy import Symbol

In [2]:
from pathlib import Path

REPO_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "hierarch_inf").is_dir()
)
APPLICATION_DIR = REPO_ROOT / "examples"
MODEL_DIR = APPLICATION_DIR / "models/generated"
DNA_NM_TO_UM = 1e-3


import sys
sys.path.insert(0, str(REPO_ROOT))


## Construct the CRN with BioCRNpyler

In [3]:
# Model choices for CRN generation.
INITIAL_DNA_CONCENTRATION = 1.99 * DNA_NM_TO_UM
ENZYME_TOTALS = {"P_tot": 0.1, "R_tot": 1.2, "E_tot": 0.001}
MODEL_PARAMETERS = {
    ("transcription_mm", None, "kb"): 10.0,
    ("transcription_mm", None, "ku"): 1.0,
    ("translation_mm", None, "kb"): 10.0,
    ("translation_mm", None, "ku"): 1.0,
    ("rna_degradation_mm", None, "kb"): 10.0,
    ("rna_degradation_mm", None, "ku"): 1.0,
    "ktx": 1.0, "ktl": 1.0, "kdil": 0.1,
    ("rna_degradation_mm", None, "kdeg"): 0.1,
}

RAW_SBML = Path(MODEL_DIR / "biocrnpyler_gfp_expression.xml")

CLEAN_SBML = Path(MODEL_DIR / "gfp_expression_bioscrape.xml")

RAW_SBML.parent.mkdir(
    parents=True,
    exist_ok=True,
)

gfp_dna = bcp.DNAassembly(
    name="gfp",
    promoter="pconst",
    rbs="rbs_strong",
    protein="GFP",
)

degradation = bcp.Dilution(
    mechanism_type="global_mechanism",
    filter_dict={"GFP": True},
    default_on=False,
)


initial_conditions = {
    "dna_gfp": INITIAL_DNA_CONCENTRATION,
    "protein_RNAP": ENZYME_TOTALS["P_tot"],
    "protein_Ribo": ENZYME_TOTALS["R_tot"],
    "protein_RNase": ENZYME_TOTALS["E_tot"],
}


txtl_extract = bcp.TxTlExtract(
    name="txtl",
    components=[gfp_dna],
    parameter_file=None,
    parameters=MODEL_PARAMETERS,
    global_mechanisms=[degradation],
)


gfp_crn = txtl_extract.compile_crn()
# The current BioCRNpyler API expects Species keys for initial concentrations.
gfp_crn.initial_concentration_dict = {
    species: initial_conditions[str(species)]
    for species in gfp_crn.species if str(species) in initial_conditions
}


gfp_crn.write_sbml_file(
    str(RAW_SBML)
)


True

Load in AutoReduce

In [4]:
species_rename = {
    "dna_gfp":
        "DNA",

    "protein_RNAP":
        "RNAP",

    "rna_gfp":
        "mRNA",

    "complex_dna_gfp_protein_RNAP_":
        "DNA_RNAP",

    "protein_Ribo":
        "Ribo",

    "protein_GFP":
        "GFP",

    "complex_protein_Ribo_rna_gfp_":
        "Ribo_mRNA",

    "protein_RNase":
        "RNase",

    "complex_protein_RNase_rna_gfp_":
        "RNase_mRNA",
}


system = load_sbml(
    str(RAW_SBML),
    outputs=["protein_GFP"],
    rename_species=species_rename,
)

Your output 'protein_GFP' is now set using the system.C matrix!


In [5]:
parameter_rename = {
    Symbol('kb__transcription_mm'): Symbol('kb_tx'),
    Symbol('ku__transcription_mm'): Symbol('ku_tx'),
    Symbol('kb__translation_mm'): Symbol('kb_tl'),
    Symbol('ku__translation_mm'): Symbol('ku_tl'),
    Symbol('kb__rna_degradation_mm'): Symbol('kb_deg'),
    Symbol('ku__rna_degradation_mm'): Symbol('ku_deg'),
    Symbol('ktx__'): Symbol('ktx'),
    Symbol('ktl__'): Symbol('ktl'),
    Symbol('kdil__'): Symbol('kdil'),
    Symbol('kdeg__rna_degradation_mm'): Symbol('kdeg'),
}

system.f = [
    fi.subs(parameter_rename)
    for fi in system.f
]


system.params_dict = {
    parameter_rename.get(param, param): value

    for param, value
    in system.params_dict.items()
}


system.params = list(
    system.params_dict.keys()
)

system.params_values = list(
    system.params_dict.values()
)

# Initial concentrations are inherited from the BioCRNpyler export.
system.write_sbml(str(CLEAN_SBML))


<SBMLDocument>

In [6]:
system.f

[-DNA*RNAP*kb_tx + DNA_RNAP*ktx + DNA_RNAP*ku_tx,
 -DNA*RNAP*kb_tx + DNA_RNAP*ktx + DNA_RNAP*ku_tx,
 DNA_RNAP*ktx - RNase*kb_deg*mRNA + RNase_mRNA*ku_deg - Ribo*kb_tl*mRNA + Ribo_mRNA*ktl + Ribo_mRNA*ku_tl,
 DNA*RNAP*kb_tx - DNA_RNAP*ktx - DNA_RNAP*ku_tx,
 -Ribo*kb_tl*mRNA + Ribo_mRNA*ktl + Ribo_mRNA*ku_tl,
 -GFP*kdil + Ribo_mRNA*ktl,
 Ribo*kb_tl*mRNA - Ribo_mRNA*ktl - Ribo_mRNA*ku_tl,
 -RNase*kb_deg*mRNA + RNase_mRNA*kdeg + RNase_mRNA*ku_deg,
 RNase*kb_deg*mRNA - RNase_mRNA*kdeg - RNase_mRNA*ku_deg]

In [7]:
system.x_init

[0.00199, 0.1, 0.0, 0.0, 1.2, 0.0, 0.0, 0.001, 0.0]

In [8]:
system.x

[DNA, RNAP, mRNA, DNA_RNAP, Ribo, GFP, Ribo_mRNA, RNase, RNase_mRNA]

In [9]:
from autoreduce import solve_conservation_laws, find_conserved_sets
from sympy import Symbol, simplify, latex
from IPython.display import Math, display

conserved_sets = find_conserved_sets(system, search_depth=2)
from autoreduce import solve_conservation_laws
from sympy import Symbol

P_set = next(
    s for s in conserved_sets
    if {str(x) for x in s}
    == {"RNAP", "DNA_RNAP"}
)

R_set = next(
    s for s in conserved_sets
    if {str(x) for x in s}
    == {"Ribo", "Ribo_mRNA"}
)

E_set = next(
    s for s in conserved_sets
    if {str(x) for x in s}
    == {"RNase", "RNase_mRNA"}
)


conserved_system_TX = solve_conservation_laws(
    system,

    total_quantities=ENZYME_TOTALS,

    conserved_sets=[
        P_set,
        R_set,
        E_set,
    ],

    states_to_eliminate=[
        Symbol("RNAP"),
        Symbol("Ribo"),
        Symbol("RNase"),
    ],
)

In [10]:
from autoreduce import solve_timescale_separation

G = Symbol("DNA")
T = Symbol("mRNA")
X = Symbol("GFP")

C1 = Symbol("DNA_RNAP")
C2 = Symbol("Ribo_mRNA")
C3 = Symbol("RNase_mRNA")


reduced_GTX, fast_subsystem = solve_timescale_separation(
    conserved_system_TX,

    slow_states=[
        G,
        T,
        X,
    ],

    fast_states=[
        C1,
        C2,
        C3,
    ],
)

Successful solution obtained with states: [DNA, mRNA, GFP]!


In [11]:
f_T = simplify(
    reduced_GTX.f[
        reduced_GTX.x.index(T)
    ]
)

f_X = simplify(
    reduced_GTX.f[
        reduced_GTX.x.index(X)
    ]
)


display(
    Math(
        r"\begin{aligned}"
        + rf"\dot{{T}} &= {latex(f_T)} \\"
        + rf"\dot{{X}} &= {latex(f_X)}"
        + r"\end{aligned}"
    )
)

<IPython.core.display.Math object>

In [12]:
from sympy import symbols, factor, together

kb_tx, ku_tx, kb_tl, ku_tl, kb_deg, ku_deg = symbols(
    "kb_tx ku_tx kb_tl ku_tl kb_deg ku_deg")
ktx, ktl, kdeg, kdil = symbols("ktx ktl kdeg kdil")
K0, K1, K2 = symbols("K_0 K_1 K_2", positive=True)

# K_1 = (ku_tx + ktx)/kb_tx, K_0 = (ku_tl + ktl)/kb_tl,
# K_2 = (ku_deg + kdeg)/kb_deg.
def apply_lumped_parameters(expr):
    """Replace kinetic parameter combinations with the existing lumped symbols."""
    return simplify(factor(together(expr.subs({
        ku_tx: kb_tx * K1 - ktx,
        ku_tl: kb_tl * K0 - ktl,
        ku_deg: kb_deg * K2 - kdeg,
    }, simultaneous=True))))

f_T_lumped = apply_lumped_parameters(f_T)
f_X_lumped = apply_lumped_parameters(f_X)
display(Math(
    r"\begin{aligned}"
    + rf"\dot{{T}} &= {latex(f_T_lumped)} \\"
    + rf"\dot{{X}} &= {latex(f_X_lumped)}"
    + r"\end{aligned}"
))


<IPython.core.display.Math object>

Clean up the symbolic equations before generating the SBML

In [13]:
from sympy import (
    collect,
    expand,
    factor,
    fraction,
    cancel,
    simplify,
    latex,
)

# ------------------------------------------------------------
# T equation
# ------------------------------------------------------------

num_T, den_T = fraction(
    cancel(f_T_lumped)
)

terms_T = collect(
    expand(num_T),
    [
        Symbol("P_tot") * ktx,
        Symbol("E_tot") * kdeg,
    ],
    evaluate=False,
)


production_T = cancel(
    (Symbol("P_tot") * ktx)
    * factor(
        terms_T[Symbol("P_tot") * ktx]
    )
    / den_T
)

degradation_T = cancel(
    (Symbol("E_tot") * kdeg)
    * factor(
        terms_T[Symbol("E_tot") * kdeg]
    )
    / den_T
)

f_T_pretty = (
    production_T
    + degradation_T
)


# ------------------------------------------------------------
# X equation
# ------------------------------------------------------------

num_X, den_X = fraction(
    cancel(f_X_lumped)
)

terms_X = collect(
    expand(num_X),
    [
        Symbol("R_tot") * ktl,
        Symbol("GFP") * kdil,
    ],
    evaluate=False,
)


production_X = cancel(
    (Symbol("R_tot") * ktl)
    * factor(
        terms_X[Symbol("R_tot") * ktl]
    )
    / den_X
)

dilution_X = cancel(
    (Symbol("GFP") * kdil)
    * factor(
        terms_X[Symbol("GFP") * kdil]
    )
    / den_X
)

f_X_pretty = (
    production_X
    + dilution_X
)


# Display
display(
    Math(
        r"\begin{aligned}"
        + rf"\dot{{T}} &= {latex(f_T_pretty)} \\"
        + rf"\dot{{X}} &= {latex(f_X_pretty)}"
        + r"\end{aligned}"
    )
)

<IPython.core.display.Math object>

In [14]:
from autoreduce import create_system
from sympy import Symbol


T = Symbol("mRNA")
X = Symbol("GFP")

DNA = Symbol("DNA")

ktx = Symbol("ktx")
K1 = Symbol("K_1")
K2 = Symbol("K_2")
kdeg = Symbol("kdeg")
ktl = Symbol("ktl")
kdil = Symbol("kdil")
K0 = Symbol("K_0")

P_tot = Symbol("P_tot")
R_tot = Symbol("R_tot")
E_tot = Symbol("E_tot")


nominal_reduced = {
    ktx: MODEL_PARAMETERS["ktx"],
    K1: (MODEL_PARAMETERS[("transcription_mm", None, "ku")] + MODEL_PARAMETERS["ktx"]) / MODEL_PARAMETERS[("transcription_mm", None, "kb")],
    K2: (MODEL_PARAMETERS[("rna_degradation_mm", None, "ku")] + MODEL_PARAMETERS[("rna_degradation_mm", None, "kdeg")]) / MODEL_PARAMETERS[("rna_degradation_mm", None, "kb")],
    kdeg: MODEL_PARAMETERS[("rna_degradation_mm", None, "kdeg")],
    ktl: MODEL_PARAMETERS["ktl"],
    kdil: MODEL_PARAMETERS["kdil"],
    K0: (MODEL_PARAMETERS[("translation_mm", None, "ku")] + MODEL_PARAMETERS["ktl"]) / MODEL_PARAMETERS[("translation_mm", None, "kb")],

    DNA: INITIAL_DNA_CONCENTRATION,

    P_tot: ENZYME_TOTALS["P_tot"],
    R_tot: ENZYME_TOTALS["R_tot"],
    E_tot: ENZYME_TOTALS["E_tot"],
}



In [15]:
reduced_system = create_system(
    x=[
        T,
        X,
    ],

    f=[
        f_T_pretty,
        f_X_pretty,
    ],

    params=list(
        nominal_reduced.keys()
    ),

    params_values=list(
        nominal_reduced.values()
    ),

    C=[
        [0, 1],
    ],
)

In [16]:
reduced_system.x_init = [
    0.0,  # mRNA
    0.0,  # GFP
]

In [17]:
REDUCED_SBML = Path(
    MODEL_DIR / "gfp_reduced_TX.xml"
)


reduced_system.write_sbml(
    str(REDUCED_SBML)
)

<SBMLDocument>

## Mechanistic model

Use the existing BioCRNpyler construction with five polymerase and ten ribosome occupancy levels, GFP maturation, and the same nominal initial DNA concentration.


In [18]:
import pandas as pd
from IPython.display import display
from bioscrape.types import Model
from hierarch_inf.models import build_detailed_crn, export_for_bioscrape


## Model structure

This model is built using the [`multi_tx`](https://biocrnpyler.readthedocs.io/en/stable/generated/biocrnpyler.mechanisms.multi_tx.html)
and [`multi_tl`](https://biocrnpyler.readthedocs.io/en/stable/generated/biocrnpyler.mechanisms.multi_tl.html) mechanisms in BioCRNpyler. These mechanisms model reversible enzyme binding, conversion from a closed complex to an active complex, and occupancy-dependent product release.
We assume a cap of 5 polymerases per DNA and 10 ribosomes per mRNA.

Write $D_n^o$ for DNA with $n$ active polymerases, and $D_n^c$ for DNA
with $n-1$ active polymerases and one newly bound polymerase in a closed
configuration. The transcription reactions are


$D+P\rightleftharpoons D_1^c,\quad$

$D_n^o+P\rightleftharpoons D_{n+1}^c,$

$D_n^c\longrightarrow D_n^o,$

$D_n^o\longrightarrow D+nP+nT,$

$D_n^c\longrightarrow D_1^c+(n-1)P+(n-1)T.$


Binding to an occupied template uses $n=1,\ldots,N_{tx}-1$;
closed-state release uses $n=2,\ldots,N_{tx}$.
Translation has equivalent reactions with $D,P,T$ replaced by mRNA,
ribosomes, and immature GFP. Note that this model is still not mechanistic at the nucleotide-level (see Jurado et al. 2026 ACS Synthetic Biology for an even more detailed model of gene expression). 

In this model, free mRNA is degraded through the existing RNase-binding mechanism. Bound RNA is protected until released, and GFP maturation and loss are modeled as:

$G_u\xrightarrow{k_{mat}}G_f,$

$G_u\xrightarrow{k_{dil}}\varnothing,$

$G_f\xrightarrow{k_{dil}}\varnothing.$

The measured output is $G_f$.

In [19]:
TX_MAX_OCCUPANCY = 5
TL_MAX_OCCUPANCY = 10
ENZYME_TOTALS = {"RNAP": 0.1, "Ribo": 1.2, "RNase": 0.001}  # µM
FIXED_RNA_PARAMETERS = {"kb_deg": 10.0, "ku_deg": 1.0, "kdeg": 0.1}

# Reference rates for forward simulation, not inferred estimates.
NOMINAL_PARAMETERS = {
    "kb_tx": 10.0, "ku_tx": 1.0, "kiso_tx": 10.0, "ktx": 1.0,
    "kb_tl": 10.0, "ku_tl": 1.0, "kiso_tl": 10.0, "ktl": 1.0,
    "kdil": 0.1,
    "kmat": 1.0 / 8.25,
    **FIXED_RNA_PARAMETERS,
}
parameter_notes = {
    "kb_tx": "RNAP association with DNA",
    "ku_tx": "RNAP dissociation from a closed complex",
    "kiso_tx": "transcription complex activation",
    "ktx": "release from a transcription complex",
    "kb_tl": "ribosome association with mRNA",
    "ku_tl": "ribosome dissociation from a closed complex",
    "kiso_tl": "translation complex activation",
    "ktl": "release from a translation complex",
    "kdil": "GFP loss (immature and fluorescent)",
    "kmat": "GFP maturation",
    "kb_deg": "RNase association with free mRNA",
    "ku_deg": "RNase dissociation",
    "kdeg": "RNase-mediated mRNA degradation",
}
display(pd.DataFrame([
    {"parameter": name, "reference value": value,
     "units": "µM⁻¹ min⁻¹" if name.startswith("kb_") else "min⁻¹",
     "meaning": parameter_notes[name]}
    for name, value in NOMINAL_PARAMETERS.items()
]))

,parameter,reference value,units,meaning
0,kb_tx,10.000000,µM⁻¹ min⁻¹,RNAP association with DNA
1,ku_tx,1.000000,min⁻¹,RNAP dissociation from a closed complex
2,kiso_tx,10.000000,min⁻¹,transcription complex activation
3,ktx,1.000000,min⁻¹,release from a transcription complex
4,kb_tl,10.000000,µM⁻¹ min⁻¹,ribosome association with mRNA
5,ku_tl,1.000000,min⁻¹,ribosome dissociation from a closed complex
6,kiso_tl,10.000000,min⁻¹,translation complex activation
7,ktl,1.000000,min⁻¹,release from a translation complex
8,kdil,0.100000,min⁻¹,GFP loss (immature and fluorescent)
9,kmat,0.121212,min⁻¹,GFP maturation


In [20]:
RAW_SBML = MODEL_DIR / "exp3_gfp_biocrnpyler.xml"
DETAILED_SBML = MODEL_DIR / "exp3_gfp_bioscrape.xml"
detailed_crn = build_detailed_crn(
    NOMINAL_PARAMETERS, TX_MAX_OCCUPANCY, TL_MAX_OCCUPANCY,
    ENZYME_TOTALS, INITIAL_DNA_CONCENTRATION,
)
species_rename = export_for_bioscrape(detailed_crn, RAW_SBML, DETAILED_SBML)
simulation_model = Model(sbml_filename=str(DETAILED_SBML))
print("Concentration states:", len(detailed_crn.species))
print("Reaction schemas (reversible bindings counted once):", len(detailed_crn.reactions))
print("Exported kinetic parameters:", list(simulation_model.get_parameter_dictionary()))
print("Model:", DETAILED_SBML)

Concentration states: 38
Reaction schemas (reversible bindings counted once): 63
Exported kinetic parameters: ['kb_tx', 'ku_tx', 'kiso_tx', 'ktx', 'kb_tl', 'ku_tl', 'kiso_tl', 'ktl', 'kdil', 'kb_deg', 'ku_deg', 'kdeg', 'kmat']
Model: c:\Users\ayush\Box\Research\hierarchical-validation-id\hierarch-inf\examples\models\generated\exp3_gfp_bioscrape.xml


In [21]:
# end